# LSTM推流预测实验 - HydroArray作业

## 实验目标

使用LSTM神经网络预测徐六泾水文站（位于长江下游）未来48小时的流量。

### 评分标准
1. 数据预处理（数据加载、划分训练/验证/测试集）
2. 模型构建（LSTM层、全连接层、dropout等）
3. 模型训练（损失函数、优化器、学习率、训练循环）
4. 结果可视化（水位/流量预测结果对比、模型评估指标）
5. 模型预测（预测2023年8月23日至31日）
6. 分析与思考

## 1. 使用HydroArray框架（仅需5行代码）

In [ ]:
# 导入HydroArray数据集接口
from HydroArray.datasets import RiverDataset

# 加载训练数据
train_ds = RiverDataset("./data/2023年6月-8月数据汇总_lstm.xlsx",
                        start="20230601", end="20230812")

print(f"训练样本数: {len(train_ds)}")
print(f"输入维度: {train_ds.input_dim}")

## 2. 加载验证数据

In [ ]:
# 加载验证数据（使用训练集的scaler保持一致）
val_ds = RiverDataset("./data/2023年6月-8月数据汇总_lstm.xlsx",
                      start="20230813", end="20230822",
                      scaler=train_ds.scaler)

print(f"验证样本数: {len(val_ds)}")

## 3. 一行代码训练模型！

In [ ]:
# 使用RiverDataset内置的训练接口
model, results = train_ds.train(
    model="lstm",
    epochs=50,
    batch_size=16,
    learning_rate=0.0001,
    hidden_dim=256,
    val_dataset=val_ds
)

In [ ]:
# 可视化训练过程
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(results['train_losses'], label='训练损失')
plt.plot(results['val_losses'], label='验证损失')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('训练损失曲线')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('./outputs/pngs/training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 模型预测与评估

In [ ]:
# 加载测试数据
test_ds = RiverDataset("./data/2023年6月-8月数据汇总_lstm.xlsx",
                       start="20230823", end="20230831",
                       scaler=train_ds.scaler)

# 预测
predictions, targets = test_ds.predict(model)
print(f"预测结果形状: {predictions.shape}")

In [ ]:
# 评估指标
from HydroArray.utils.metrics import nse, rmse, mae

metrics = {
    'NSE': nse(predictions, targets),
    'RMSE': rmse(predictions, targets),
    'MAE': mae(predictions, targets)
}

print("\n========== 评估指标 ==========")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")
print("================================")

In [ ]:
# 可视化预测结果
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# 时序对比图
ax1 = axes[0]
n_samples = min(500, len(predictions))
t = np.arange(n_samples)
ax1.plot(t, targets[:n_samples, 0], 'b-', label='实测流量', alpha=0.7, linewidth=1)
ax1.plot(t, predictions[:n_samples, 0], 'r--', label='预测流量', alpha=0.7, linewidth=1)
ax1.set_xlabel('时间步 (30min)')
ax1.set_ylabel('流量 (m³/s)')
ax1.set_title(f'徐六泾水文站LSTM流量预测结果 (NSE={metrics["NSE"]:.3f})')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 散点图
ax2 = axes[1]
ax2.scatter(targets.flatten(), predictions.flatten(), alpha=0.3, s=10)
lim = [min(targets.min(), predictions.min()), max(targets.max(), predictions.max())]
ax2.plot(lim, lim, 'r--', label='1:1线')
ax2.set_xlabel('实测流量 (m³/s)')
ax2.set_ylabel('预测流量 (m³/s)')
ax2.set_title('预测vs实测散点图')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./outputs/pngs/prediction_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 分析与思考

### 5.1 模型性能分析

根据实验结果，分析LSTM模型在长江下游径流预测中的表现：

1. **NSE (Nash-Sutcliffe Efficiency Coefficient)**: 
   - NSE=1表示完美预测，NSE=0表示模型预测与均值相当
   - NSE > 0.5 通常被认为是可接受的性能

2. **RMSE (Root Mean Square Error)**: 
   - RMSE越小表示预测误差越小

### 5.2 长江下游水文特点

1. **潮汐影响显著**: 长江下游受潮汐影响，涨潮落潮交替
2. **双向水流**: 感潮河段水流方向随潮汐变化
3. **复杂性**: 多种因素（上游来水、潮汐、气象等）共同影响

### 5.3 改进方向

1. 尝试双向LSTM捕捉潮汐双向特征
2. 加入Attention机制增强长期依赖
3. 考虑物理约束（如水量平衡）

In [ ]:
print("=" * 60)
print("实验总结:")
print(f"  - 训练数据: 20230601 ~ 20230812")
print(f"  - 验证数据: 20230813 ~ 20230822")
print(f"  - 测试数据: 20230823 ~ 20230831")
print(f"  - 模型: Encoder-Decoder LSTM")
print(f"  - 输入序列: 48步 (24小时)")
print(f"  - 预测步长: 48步 (24小时)")
print("\n评估指标:")
for name, value in metrics.items():
    print(f"  - {name}: {value:.4f}")
print("=" * 60)
print("\n实验完成！HydroArray让AI水文预测变得简单。")